### Naseem Saleh
### INST414
### Module 2 Assignment
### October 3, 2025

cite:
(1) Robert West and Jure Leskovec:
    Human Wayfinding in Information Networks.
    _21st International World Wide Web Conference (WWW)_, 2012.
    
(2) Robert West, Joelle Pineau, and Doina Precup:
    Wikispeedia: An Online Game for Inferring Semantic Distances between Concepts.
    _21st International Joint Conference on Artificial Intelligence (IJCAI)_, 2009.

Robert West and Jure Leskovec: Human Wayfinding in Information Networks. _21st International World Wide Web Conference (WWW)_, 2012.

Robert West, Joelle Pineau, and Doina Precup: Wikispeedia: An Online Game for Inferring Semantic Distances between Concepts. _21st International Joint Conference on Artificial Intelligence (IJCAI)_, 2009.

https://snap.stanford.edu/data/wikispeedia.html


Question - What articles have the most links in the network, and what kinds of articles are they linked to?

Getting Betweenness Centrality metric for Wikispeedia dataset

In [29]:
import pandas as pd

import networkx as nx
import matplotlib.pyplot as plt

In [30]:
#Looking at the file's formatting (also did outside of python though)
with open("Documents\\wikispeedia_paths-and-graph\\links.tsv", "r", encoding = "utf-8") as f:
    for line in f:
        lines = f.readlines()

        print(lines[:14])

['# Article names are URL-encoded; e.g., in Java they can be decoded using java.net.URLDecoder.decode(articleName, "UTF-8").\n', '# FORMAT:   linkSource   linkTarget\n', '#\n', '# When publishing on this data set, please cite:\n', '# (1) Robert West and Jure Leskovec:\n', '#     Human Wayfinding in Information Networks.\n', '#     21st International World Wide Web Conference (WWW), 2012.\n', '# (2) Robert West, Joelle Pineau, and Doina Precup:\n', '#     Wikispeedia: An Online Game for Inferring Semantic Distances between Concepts.\n', '#     21st International Joint Conference on Artificial Intelligence (IJCAI), 2009.\n', '\n', '%C3%81ed%C3%A1n_mac_Gabr%C3%A1in\tBede\n', '%C3%81ed%C3%A1n_mac_Gabr%C3%A1in\tColumba\n', '%C3%81ed%C3%A1n_mac_Gabr%C3%A1in\tD%C3%A1l_Riata\n']


In [33]:
#Loads into a pandas dataframe for easier parsing, Seperates on tabs, Skips the comments
wiki_links_df = pd.read_csv("Documents\\wikispeedia_paths-and-graph\\links.tsv", sep = "\t", comment = "#", names = ["link_source", "link_target"])

print("Number of Edges", len(wiki_links_df))
#Print to check first 5 rows of data:
print(wiki_links_df.head())

Number of Edges 119882
                        link_source     link_target
0  %C3%81ed%C3%A1n_mac_Gabr%C3%A1in            Bede
1  %C3%81ed%C3%A1n_mac_Gabr%C3%A1in         Columba
2  %C3%81ed%C3%A1n_mac_Gabr%C3%A1in  D%C3%A1l_Riata
3  %C3%81ed%C3%A1n_mac_Gabr%C3%A1in   Great_Britain
4  %C3%81ed%C3%A1n_mac_Gabr%C3%A1in         Ireland


In [37]:
#Importing to clean the link names to be readable:
from urllib.parse import unquote

#Cleaning the names:
wiki_links_df["link_source"] = wiki_links_df["link_source"].apply(unquote)
wiki_links_df["link_target"] = wiki_links_df["link_target"].apply(unquote)

#Checking it was done correctly
print("Number of Edges", len(wiki_links_df))
#Print to check first 5 rows of data:
print("\n", wiki_links_df.head())

Number of Edges 119882

          link_source    link_target
0  Áedán_mac_Gabráin           Bede
1  Áedán_mac_Gabráin        Columba
2  Áedán_mac_Gabráin      Dál_Riata
3  Áedán_mac_Gabráin  Great_Britain
4  Áedán_mac_Gabráin        Ireland


In [129]:
#Loading categories into df to understand the themes in the data
wiki_categories_df = pd.read_csv("Documents\\wikispeedia_paths-and-graph\\categories.tsv", sep="\t", comment="#", names=["article", "category"])

#Cleaning the names:
wiki_categories_df["article"] = wiki_categories_df["article"].apply(unquote)
wiki_categories_df["category"] = wiki_categories_df["category"].apply(unquote)

categories_dict = wiki_categories_df.groupby("article")["category"].apply(list).to_dict()


#Checking the categories
print("Number of Article-Category Pairs:", len(wiki_categories_df))
print("\nFirst 10 Categories:")
print(wiki_categories_df.head(10))

# Seeing how many unique categories there are:
print("\nNumber of Categories: ", len(wiki_categories_df['category'].value_counts()))


##Adds the categories to the attributes for nodes
nx.set_node_attributes(wiki_graph, categories_dict, "categories")


#Checking the categories are correctly added:
print("\n\n Checking Categories in Graph:")
for node in list(wiki_graph.nodes())[:10]:
    print(node, wiki_graph.nodes[node].get("categories"))


Number of Article-Category Pairs: 5204

First 10 Categories:
                  article                                           category
0       Áedán_mac_Gabráin  subject.History.British_History.British_Histor...
1       Áedán_mac_Gabráin                  subject.People.Historical_figures
2                   Åland                                  subject.Countries
3                   Åland  subject.Geography.European_Geography.European_...
4           Édouard_Manet                             subject.People.Artists
5                    Éire                                  subject.Countries
6                    Éire  subject.Geography.European_Geography.European_...
7   Óengus_I_of_the_Picts  subject.History.British_History.British_Histor...
8   Óengus_I_of_the_Picts                  subject.People.Historical_figures
9  €2_commemorative_coins                  subject.Business_Studies.Currency

Number of Categories:  129


 Checking Categories in Graph:
Áedán_mac_Gabráin ['subject.His

In [56]:
#Creating the directed graph for the article links
wiki_graph = nx.from_pandas_edgelist(wiki_links_df, source='link_source', target='link_target', create_using=nx.DiGraph())

#Checking number of nodes and edges
print(wiki_graph.number_of_nodes())
print(wiki_graph.number_of_edges())

4592
119882


In [19]:
#Checking the amount of edges or nodes
print("Number of Nodes", len(wiki_graph.nodes()))
print("Number of Edges", len(wiki_graph.edges()))

Number of Nodes 4592
Number of Edges 119882


In [17]:
print("Nodes:", len(wiki_graph.nodes))

Nodes: 4592


In [96]:
## Performing the betweeness centrality method on the data to analyze:

top_k_nodes = 10

# Getting the top 10 ?? for Betweenness Centrality measure
centrality_betweenness = nx.betweenness_centrality(wiki_graph)
print("Top 10 by Betweenness")
for w_article in sorted(centrality_betweenness, key=centrality_betweenness.get, reverse=True)[:top_k_nodes]:
    print(w_article, wiki_graph.nodes[w_article], centrality_betweenness[w_article])

#wiki_graph.nodes  gets the categories attributes

Top 10 by Betweenness
United_States {'categories': ['subject.Countries', 'subject.Geography.North_American_Geography']} 0.09409037321391792
United_Kingdom {'categories': ['subject.Countries', 'subject.Geography.European_Geography.European_Countries', 'subject.Geography.Geography_of_Great_Britain']} 0.04238444878433579
England {'categories': ['subject.Geography.Geography_of_Great_Britain']} 0.03240281285403478
Europe {'categories': ['subject.Geography.European_Geography']} 0.026991699473661448
Africa {'categories': ['subject.Geography.African_Geography']} 0.02420417619015421
Germany {'categories': ['subject.Countries', 'subject.Geography.European_Geography.European_Countries']} 0.019408600675210556
World_War_II {'categories': ['subject.History.British_History.British_History_Post_1900']} 0.015499598553131516
19th_century {'categories': ['subject.History.General_history']} 0.014823815013166167
London {'categories': ['subject.Geography.Geography_of_Great_Britain']} 0.014675371952972789
En

In [122]:
#Printing to check edges for possible 3 chosen query nodes
for node in ["Africa", "19th_century", "English_language"]:
    print(f"Edges for {node}:")
    print(list(wiki_graph.edges(node)), "\n")

Edges for Africa:
[('Africa', 'Abidjan'), ('Africa', 'Abuja'), ('Africa', 'Accra'), ('Africa', 'Addis_Ababa'), ('Africa', 'African_Great_Lakes'), ('Africa', 'African_Union'), ('Africa', 'Afrikaans'), ('Africa', 'Agriculture'), ('Africa', 'Algeria'), ('Africa', 'Algiers'), ('Africa', 'Ancient_Egypt'), ('Africa', 'Ancient_Greece'), ('Africa', 'Ancient_Rome'), ('Africa', 'Anglican_Communion'), ('Africa', 'Angola'), ('Africa', 'Antananarivo'), ('Africa', 'Anthropology'), ('Africa', 'Arabic_language'), ('Africa', 'Asia'), ('Africa', 'Atlantic_Ocean'), ('Africa', 'Atlantic_slave_trade'), ('Africa', 'Bamako'), ('Africa', 'Banjul'), ('Africa', 'Bantu'), ('Africa', 'Belgium'), ('Africa', 'Benin'), ('Africa', 'Bissau'), ('Africa', 'Botswana'), ('Africa', 'Burkina_Faso'), ('Africa', 'Burundi'), ('Africa', "Côte_d'Ivoire"), ('Africa', 'Cairo'), ('Africa', 'Camel'), ('Africa', 'Cameroon'), ('Africa', 'Cape_Town'), ('Africa', 'Cape_Verde'), ('Africa', 'Capital'), ('Africa', 'Carnivore'), ('Africa', 

In [113]:
#Chosen 3 query nodes:
query_nodes =  ["Africa", "19th_century", "English_language"]

In [138]:
#Creating a subset graph with neighbors for Africa, 19th_century, and English_language articles
neighbor_nodes = set(query_nodes)

#Looping through nodes to get their neighboring edges
for node in query_nodes:
    if node in wiki_graph:
        neighbor_nodes.update(wiki_graph.successors(node))     # Linked from query node to neighbor
        neighbor_nodes.update(wiki_graph.predecessors(node))   # Linked from neighbor to query node

#Creating the sub graph and copying it so can change the attributes
sub_graph = wiki_graph.subgraph(neighbor_nodes).copy()

for node in sub_graph.nodes():
    #Adds query tags to highlight the query nodes in Gephi:
    if node in query_nodes:
        sub_graph.nodes[node]["is_query"] = "True"
    else:
        sub_graph.nodes[node]["is_query"] = "False"

    #To fix type error, categories originally in list, won't be accepted by Gephi Graphml
    for attribute in sub_graph.nodes[node]:
        if isinstance(sub_graph.nodes[node][attribute], list):
            sub_graph.nodes[node][attribute] = ", ".join(sub_graph.nodes[node][attribute])

#Creating the graph for Gephi  graphml  for visualization
nx.write_graphml(sub_graph, "Documents\\wikispeedia_analysis_subgraph1.graphml")

In [ ]:
# Or full graph

#nx.write_graphml(wiki_graph, "wikispeedia_analysis_Full_graph.graphml")

In [139]:
#Making a graph with smaller set of neighbors (20):

neighbor_nodes = set(query_nodes)

#Looping through nodes to get their neighboring edges
for node in query_nodes:
    if node in wiki_graph:
        successors = list(wiki_graph.successors(node))[:20] # Linked from query node to neighbor
        predecessors = list(wiki_graph.predecessors(node))[:20] # Linked from neighbor to query node
        neighbor_nodes.update(successors + predecessors)

#Creating the sub graph and copying it so can change the attributes
sub_graph2 = wiki_graph.subgraph(neighbor_nodes).copy()

for node in sub_graph2.nodes():
    #Adds query tags to highlight the query nodes in Gephi:
    if node in query_nodes:
        sub_graph2.nodes[node]["is_query"] = "True"
    else:
        sub_graph2.nodes[node]["is_query"] = "False"

    #To fix type error, categories originally in list, won't be accepted by Gephi Graphml
    for attribute in sub_graph2.nodes[node]:
        if isinstance(sub_graph2.nodes[node][attribute], list):
            sub_graph2.nodes[node][attribute] = ", ".join(sub_graph2.nodes[node][attribute])

#Creating the graph for Gephi  graphml  for visualization
nx.write_graphml(sub_graph2, "Documents\\wikispeedia_analysis_subgraph_2_smaller.graphml")